# CivicLens: 03. Semantic Similarity and Deduplication
This notebook benchmarks Lexical, Semantic, and Hybrid similarity formulations for detecting duplicate reports across paraphrase and code-mixed inputs.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt
from src.similarity import HybridSimilarityEngine
from src.visualization import plot_similarity_score_distributions

## 1. Load Labeled Complaint Pairs

In [ ]:
pairs_df = pd.read_csv('../data/complaint_pairs.csv')
complaints_df = pd.read_csv('../data/sample_complaints.csv')
print(f'Total pairwise test cases: {len(pairs_df)}')
pairs_df['pair_type'].value_counts()

## 2. Fit Hybrid Engine and Optimize Decision Threshold

In [ ]:
engine = HybridSimilarityEngine().fit_corpus(complaints_df['text'].tolist())
opt_tau, best_f1 = engine.optimize_threshold(pairs_df, alpha=0.70)
print(f'Optimal decision threshold tau*: {opt_tau:.2f} | Best F1: {best_f1:.4f}')

## 3. Comparative Evaluation: Lexical vs Semantic vs Hybrid

In [ ]:
res_lex = engine.evaluate_pairs(pairs_df, alpha=0.0, threshold=0.25)
res_sem = engine.evaluate_pairs(pairs_df, alpha=1.0, threshold=0.60)
res_hyb = engine.evaluate_pairs(pairs_df, alpha=0.70, threshold=opt_tau)

comparison = pd.DataFrame([
    {'Model': 'Lexical Only (alpha=0.0)', 'Accuracy': res_lex['accuracy'], 'Precision': res_lex['precision'], 'Recall': res_lex['recall'], 'F1': res_lex['f1_score']},
    {'Model': 'Semantic Only (alpha=1.0)', 'Accuracy': res_sem['accuracy'], 'Precision': res_sem['precision'], 'Recall': res_sem['recall'], 'F1': res_sem['f1_score']},
    {'Model': 'Hybrid (alpha=0.70)', 'Accuracy': res_hyb['accuracy'], 'Precision': res_hyb['precision'], 'Recall': res_hyb['recall'], 'F1': res_hyb['f1_score']}
])
comparison